# 10. 錯誤處理與重試

學習如何在 LangGraph 中處理錯誤並實現重試機制。

---

## 🎯 學習目標

完成本章節後，您將能夠：
- ✅ 設計錯誤處理流程
- ✅ 實現自動重試機制
- ✅ 建立 Fallback 備用方案
- ✅ 優雅地處理各種失敗情況

---

## 📊 錯誤處理策略

```
┌─────────────────────────────────────────────────────────┐
│                   錯誤處理策略                           │
├─────────────────────────────────────────────────────────┤
│                                                         │
│   ┌──────────────────────────────────────────────────┐ │
│   │              策略 1: 重試 (Retry)                 │ │
│   │   失敗 → 等待 → 重試 → 失敗 → 等待 → 重試...     │ │
│   └──────────────────────────────────────────────────┘ │
│                                                         │
│   ┌──────────────────────────────────────────────────┐ │
│   │              策略 2: 降級 (Fallback)              │ │
│   │   主服務失敗 → 切換到備用服務                     │ │
│   └──────────────────────────────────────────────────┘ │
│                                                         │
│   ┌──────────────────────────────────────────────────┐ │
│   │              策略 3: 優雅降級                     │ │
│   │   失敗 → 返回預設值/快取                          │ │
│   └──────────────────────────────────────────────────┘ │
│                                                         │
└─────────────────────────────────────────────────────────┘
```

In [1]:
import random
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

---

## 10.1 基本錯誤捕獲

### 在節點中處理錯誤

```python
def my_node(state):
    try:
        # 可能失敗的操作
        result = risky_operation()
        return {"result": result, "error": None}
    except Exception as e:
        return {"result": None, "error": str(e)}
```

In [2]:
class ErrorState(TypedDict):
    """錯誤處理狀態"""
    input: str
    result: str
    error: str | None

def risky_operation(state: ErrorState) -> dict:
    """可能失敗的操作（50% 失敗率）"""
    try:
        if random.random() < 0.5:
            raise ValueError("隨機錯誤發生！")
        print("  ✅ 操作成功")
        return {"result": f"成功處理: {state['input']}", "error": None}
    except Exception as e:
        print(f"  ❌ 操作失敗: {e}")
        return {"result": "", "error": str(e)}

def check_error(state: ErrorState) -> Literal["success", "error"]:
    """檢查是否有錯誤"""
    return "error" if state.get("error") else "success"

def handle_error(state: ErrorState) -> dict:
    """錯誤處理器"""
    print(f"  🔧 處理錯誤: {state['error']}")
    return {"result": f"錯誤已處理: {state['error']}", "error": None}

# 建構圖
graph = StateGraph(ErrorState)
graph.add_node("process", risky_operation)
graph.add_node("error_handler", handle_error)

graph.add_edge(START, "process")
graph.add_conditional_edges("process", check_error, {
    "success": END,
    "error": "error_handler"
})
graph.add_edge("error_handler", END)

app = graph.compile()
print("✅ 錯誤處理流程已就緒")

✅ 錯誤處理流程已就緒


In [3]:
print("📊 測試錯誤處理（執行 5 次）：")
print("=" * 50)

for i in range(5):
    print(f"\n--- 測試 {i+1} ---")
    result = app.invoke({"input": f"測試{i}", "result": "", "error": None})
    print(f"  結果: {result['result']}")

📊 測試錯誤處理（執行 5 次）：

--- 測試 1 ---
  ❌ 操作失敗: 隨機錯誤發生！
  🔧 處理錯誤: 隨機錯誤發生！
  結果: 錯誤已處理: 隨機錯誤發生！

--- 測試 2 ---
  ❌ 操作失敗: 隨機錯誤發生！
  🔧 處理錯誤: 隨機錯誤發生！
  結果: 錯誤已處理: 隨機錯誤發生！

--- 測試 3 ---
  ✅ 操作成功
  結果: 成功處理: 測試2

--- 測試 4 ---
  ❌ 操作失敗: 隨機錯誤發生！
  🔧 處理錯誤: 隨機錯誤發生！
  結果: 錯誤已處理: 隨機錯誤發生！

--- 測試 5 ---
  ✅ 操作成功
  結果: 成功處理: 測試4


---

## 10.2 重試機制

### 重試策略比較

| 策略 | 說明 | 適用場景 |
|------|------|----------|
| 固定間隔 | 每次等待相同時間 | 簡單場景 |
| 指數退避 | 每次等待時間翻倍 | 外部 API |
| 有限次數 | 設定最大重試次數 | 防止無限重試 |

In [4]:
class RetryState(TypedDict):
    """重試狀態"""
    input: str
    result: str
    attempts: int        # 當前嘗試次數
    max_attempts: int    # 最大嘗試次數
    success: bool

def attempt_operation(state: RetryState) -> dict:
    """嘗試執行操作（70% 失敗率來模擬不穩定服務）"""
    attempts = state.get("attempts", 0) + 1
    print(f"  🔄 嘗試 {attempts}/{state['max_attempts']}")
    
    if random.random() < 0.7:  # 70% 失敗率
        print("     ❌ 失敗")
        return {"attempts": attempts, "success": False}
    
    print("     ✅ 成功！")
    return {"attempts": attempts, "success": True, "result": "操作成功完成"}

def should_retry(state: RetryState) -> Literal["retry", "success", "give_up"]:
    """決定下一步行動"""
    if state.get("success"):
        return "success"
    if state["attempts"] >= state["max_attempts"]:
        return "give_up"
    return "retry"

def finalize_success(state): 
    print("  🎉 最終結果: 成功")
    return {"result": "✅ 操作成功"}

def finalize_failure(state): 
    print(f"  😢 最終結果: 放棄（達到 {state['max_attempts']} 次）")
    return {"result": f"❌ 達到最大重試次數"}

# 建構重試圖
retry_graph = StateGraph(RetryState)
retry_graph.add_node("attempt", attempt_operation)
retry_graph.add_node("success", finalize_success)
retry_graph.add_node("failure", finalize_failure)

retry_graph.add_edge(START, "attempt")
retry_graph.add_conditional_edges("attempt", should_retry, {
    "retry": "attempt",      # 🔄 重試
    "success": "success",    # ✅ 成功
    "give_up": "failure"     # ❌ 放棄
})
retry_graph.add_edge("success", END)
retry_graph.add_edge("failure", END)

retry_app = retry_graph.compile()
print("✅ 重試機制已就緒")

✅ 重試機制已就緒


In [5]:
print("📊 測試重試機制：")
print("=" * 50)

result = retry_app.invoke({
    "input": "測試任務",
    "result": "",
    "attempts": 0,
    "max_attempts": 5,
    "success": False
})

print("\n" + "=" * 50)
print(f"嘗試次數: {result['attempts']}")
print(f"最終結果: {result['result']}")

📊 測試重試機制：
  🔄 嘗試 1/5
     ❌ 失敗
  🔄 嘗試 2/5
     ✅ 成功！
  🎉 最終結果: 成功

嘗試次數: 2
最終結果: ✅ 操作成功


---

## 10.3 Fallback 機制

當主服務失敗時，切換到備用服務：

In [6]:
class FallbackState(TypedDict):
    """Fallback 狀態"""
    query: str
    result: str
    service_used: str

def primary_service(state: FallbackState) -> dict:
    """主服務（50% 失敗率）"""
    if random.random() < 0.5:
        print("  🔴 主服務失敗")
        return {"result": "__FAILED__", "service_used": "primary"}
    print("  🟢 主服務成功")
    return {"result": "主服務回應", "service_used": "primary"}

def fallback_service(state: FallbackState) -> dict:
    """備用服務（總是成功）"""
    print("  🟡 使用備用服務")
    return {"result": "備用服務回應", "service_used": "fallback"}

def check_fallback(state: FallbackState) -> Literal["done", "fallback"]:
    """檢查是否需要 fallback"""
    if state["result"] == "__FAILED__":
        return "fallback"
    return "done"

# 建構 Fallback 圖
fb_graph = StateGraph(FallbackState)
fb_graph.add_node("primary", primary_service)
fb_graph.add_node("fallback", fallback_service)

fb_graph.add_edge(START, "primary")
fb_graph.add_conditional_edges("primary", check_fallback, {
    "done": END,
    "fallback": "fallback"
})
fb_graph.add_edge("fallback", END)

fb_app = fb_graph.compile()
print("✅ Fallback 機制已就緒")

✅ Fallback 機制已就緒


In [7]:
print("📊 測試 Fallback 機制（執行 5 次）：")
print("=" * 50)

stats = {"primary": 0, "fallback": 0}

for i in range(5):
    print(f"\n--- 測試 {i+1} ---")
    r = fb_app.invoke({"query": "test", "result": "", "service_used": ""})
    stats[r["service_used"]] += 1
    print(f"  結果: {r['result']} (使用: {r['service_used']})")

print("\n" + "=" * 50)
print(f"統計: 主服務 {stats['primary']} 次, 備用 {stats['fallback']} 次")

📊 測試 Fallback 機制（執行 5 次）：

--- 測試 1 ---
  🔴 主服務失敗
  🟡 使用備用服務
  結果: 備用服務回應 (使用: fallback)

--- 測試 2 ---
  🟢 主服務成功
  結果: 主服務回應 (使用: primary)

--- 測試 3 ---
  🟢 主服務成功
  結果: 主服務回應 (使用: primary)

--- 測試 4 ---
  🔴 主服務失敗
  🟡 使用備用服務
  結果: 備用服務回應 (使用: fallback)

--- 測試 5 ---
  🟢 主服務成功
  結果: 主服務回應 (使用: primary)

統計: 主服務 3 次, 備用 2 次


---

## 💡 重點回顧

### 錯誤處理模式

```python
# 模式 1: 節點內捕獲
def node(state):
    try:
        ...
    except Exception as e:
        return {"error": str(e)}

# 模式 2: 條件路由到錯誤處理器
graph.add_conditional_edges("node", check_error, {
    "success": END,
    "error": "error_handler"
})

# 模式 3: 循環重試
graph.add_conditional_edges("node", should_retry, {
    "retry": "node"  # 回到自己
})
```

### 設計原則

| 原則 | 說明 |
|------|------|
| 限制重試 | 設定最大次數防止無限循環 |
| 記錄錯誤 | 保存錯誤資訊用於除錯 |
| 優雅降級 | 失敗時提供替代方案 |
| 快速失敗 | 某些情況下直接失敗更好 |

---

## 📝 練習題

1. **指數退避**：實作每次重試等待時間翻倍
2. **錯誤日誌**：記錄所有錯誤到 State 中
3. **多級 Fallback**：實作 3 級備用方案
4. **熔斷器**：連續失敗 N 次後暫時停止嘗試

---

下一步：[11. 串流輸出](11_streaming.ipynb)